<a href="https://colab.research.google.com/github/sujith-kumara/Design-a-Deep-Learning-Framework-GNR-638-Assignment/blob/main/Assignment_2/resnet50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [56]:
from google.colab import drive


In [57]:
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [58]:
!unzip -q "/content/drive/MyDrive/GNR638_assignment1/train_data.zip" -d /content/

replace /content/train_data/Airport/Airport_0000.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: N
N


In [60]:
pip install ptflops

In [61]:
!pip install timm torch torchvision

In [62]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import timm
import numpy as np
from ptflops import get_model_complexity_info
from sklearn.model_selection import train_test_split
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix
import pandas as pd
from torchvision.transforms import v2

## --- Configuration ---

In [63]:
DATA_DIR = "/content/train_data"
NUM_CLASSES = 30
BATCH_SIZE = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## --- Data Loading ---

In [64]:
SEED = 42

In [65]:
# Standard ImageNet transforms for pre-trained models
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## --- Dataset Splitting ---

In [66]:
# 1. Create two base datasets pointing to the SAME folder, but with different transforms
base_train_dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform_train)
base_val_dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform_val)

In [67]:
base_train_dataset

Dataset ImageFolder
    Number of datapoints: 6993
    Root location: /content/train_data
    StandardTransform
Transform: Compose(
               Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
               RandomHorizontalFlip(p=0.5)
               ToTensor()
               Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
           )

In [68]:
# 2. Get the targets (class labels) for stratification
targets = base_train_dataset.targets
indices = np.arange(len(targets))

In [69]:
# 3. Perform a stratified split (e.g., 80% train, 20% validation)
train_indices, val_indices = train_test_split(
    indices,
    test_size=0.20,
    random_state=SEED,
    stratify=targets
)

In [70]:
# 4. Create the final Subsets
train_dataset = Subset(base_train_dataset, train_indices)
val_dataset = Subset(base_val_dataset, val_indices)

In [71]:
# --- DataLoaders ---
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Total images: {len(base_train_dataset)}")
print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")

Total images: 6993
Training images: 5594
Validation images: 1399


## --- Model Selection & Efficiency Metrics ---

In [72]:
# --- Model ---
def initialize_model(model_name):
    model = timm.create_model(model_name, pretrained=True, num_classes=NUM_CLASSES)
    model = model.to(DEVICE)
    return model

In [73]:
def report_efficiency(model):
    """Reports parameters, MACs, and FLOPs."""
    macs, params = get_model_complexity_info(model, (3, 224, 224), as_strings=True, print_per_layer_stat=False)
    print(f"Parameters: {params}, MACs: {macs}")

## Experiment 1

In [18]:
def setup_linear_probe(model):
    # 1. Freeze all backbone parameters
    for param in model.parameters():
        param.requires_grad = False

    # 2. Unfreeze the final classification layer (timm uses 'get_classifier' or 'head')
    model.reset_classifier(NUM_CLASSES)

    # 3. Setup optimizer to ONLY train the linear classifier
    optimizer = torch.optim.Adam(model.get_classifier().parameters(), lr=1e-3)
    return model, optimizer

In [19]:
def run_linear_probe(model_name):
    print(f"\n{'='*50}\nStarting Scenario 4.1: Linear Probe for {model_name}\n{'='*50}")

    model = initialize_model(model_name)
    report_efficiency(model)

    # Setup for linear probe
    model, optimizer = setup_linear_probe(model)

    # Train ( 30 epochs for full data)
    print("Training Linear Probe...")
    trained_model, history = train_model(model, train_loader, val_loader, optimizer, num_epochs=2)

    # Save the weights for later visualization (PCA/t-SNE)
    torch.save(trained_model.state_dict(), f"{model_name}_linear_probe.pth")
    return trained_model, history

## Experiment 2

In [46]:
def run_finetuning_experiments(model_name="resnet50"):
    print(f"\n{'='*60}")
    print(f"Starting Scenario 4.2: Fine-Tuning Strategies for {model_name}")
    print(f"{'='*60}\n")

    # The 4 strategies required by the assignment
    # strategies = ["linear", "last_block", "selective_20", "full"]
    strategies = ["last_block", "selective_20", "full"]

    # Data structures to hold results for the final plots
    final_val_accs = []
    percent_unfrozen_list = []
    training_histories = []
    all_gradient_norms = {}

    for strategy in strategies:
        print(f"\n--- Running Strategy: {strategy.upper()} ---")

        # 1. Initialize a fresh model for each strategy to avoid leakage
        model = initialize_model(model_name)

        # 2. Setup the freezing/unfreezing based on the strategy
        model, optimizer, percent_unfrozen = setup_finetuning_strategy(model, strategy, model_name)
        percent_unfrozen_list.append(percent_unfrozen)

        # 3. Train the model
        # Note: Ensure your train_model has the `enumerate(train_loader)` fix applied!
        print(f"Training {strategy}...")
        trained_model, history = train_model(model, train_loader, val_loader, optimizer, num_epochs=2,experiment=2)

        # 4. Save metrics for plotting
        final_val_accs.append(history['val_acc'][-1])
        training_histories.append(history)

        # all_gradient_norms[strategy] = epoch_grad_norms

    # --- Generate the Required Reports ---
    print("\n--- Generating Scenario 4.2 Plots ---")
    plot_finetuning_results(strategies, percent_unfrozen_list, final_val_accs, training_histories)
    # plot_gradient_norms(all_gradient_norms)

In [47]:

def setup_finetuning_strategy(model, strategy, model_name="resnet50"):
    """
    Configures the freezing/unfreezing of model parameters based on the strategy.
    Strategies: 'linear', 'last_block', 'full', 'selective_20'
    """
    model = model.to(DEVICE)

    # 1. Start by freezing everything
    for param in model.parameters():
        param.requires_grad = False

    total_params = sum(p.numel() for p in model.parameters())
    unfrozen_params = 0

    # 2. Apply strategy
    if strategy == "linear":
        # Only unfreeze the classifier head
        model.reset_classifier(30)

    elif strategy == "full":
        # Unfreeze everything
        for param in model.parameters():
            param.requires_grad = True

    elif strategy == "last_block":
        # Unfreeze the classifier head
        model.reset_classifier(30)
        # Architecture-specific unfreezing for the "last block"
        if model_name == "resnet50":
            for param in model.layer4.parameters():
                param.requires_grad = True

    elif strategy == "selective_20":
        # Unfreeze the classifier head
        model.reset_classifier(30)
        target_unfrozen = total_params * 0.20

        # Iterate backward and unfreeze until we hit the ~20% parameter budget
        current_unfrozen = 0
        for name, param in reversed(list(model.named_parameters())):
            if current_unfrozen < target_unfrozen:
                param.requires_grad = True
                current_unfrozen += param.numel()
            else:
                break

    # Calculate exact percentage of unfrozen parameters for the report
    unfrozen_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    percent_unfrozen = (unfrozen_params / total_params) * 100

    print(f"Strategy: {strategy} | Unfrozen Params: {percent_unfrozen:.2f}%")

    # Return model and optimizer (only pass parameters that require gradients)
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
    return model, optimizer, percent_unfrozen

In [48]:
def track_gradient_norms(model):
    """Calculates the L2 norm of gradients for different logical sections of the model."""
    grad_norms = {}
    for name, param in model.named_parameters():
        if param.requires_grad and param.grad is not None:
            # Group by major blocks to keep the plot readable
            base_name = name.split('.')[0]
            norm = param.grad.data.norm(2).item()

            if base_name in grad_norms:
                grad_norms[base_name] += norm
            else:
                grad_norms[base_name] = norm
    return grad_norms

In [49]:
def plot_finetuning_results(strategy_names, percent_unfrozen_list, final_val_accs, training_histories):
    """
    Plots the specific metrics required for Scenario 4.2.
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Plot 1: Accuracy vs Percentage of Unfrozen Parameters
    axes[0].plot(percent_unfrozen_list, final_val_accs, marker='o', linestyle='-', color='b')
    for i, txt in enumerate(strategy_names):
        axes[0].annotate(txt, (percent_unfrozen_list[i], final_val_accs[i]),
                         textcoords="offset points", xytext=(0,10), ha='center')
    axes[0].set_title('Validation Accuracy vs. Unfrozen Parameters')
    axes[0].set_xlabel('Percentage of Unfrozen Parameters (%)')
    axes[0].set_ylabel('Validation Accuracy')
    axes[0].grid(True, linestyle='', alpha=0.7)

    # Plot 2: Convergence Stability (Training Loss vs Epoch)
    for strategy_name, history in zip(strategy_names, training_histories):
        epochs = range(1, len(history['train_loss']) + 1)
        axes[1].plot(epochs, history['train_loss'], label=strategy_name, marker='.')

    axes[1].set_title('Convergence Stability (Training Loss)')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Training Loss')
    axes[1].legend()
    axes[1].grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()

## Experiment 3

In [50]:
def get_few_shot_dataloaders(dataset, percentage, seed=42):
    dataset_size = len(dataset)
    indices = list(range(dataset_size))
    split = int(np.floor(percentage * dataset_size))

    np.random.seed(seed)
    np.random.shuffle(indices)

    subset_indices = indices[:split]
    subset = Subset(dataset, subset_indices)

    return DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True)


In [51]:
def run_few_shot_trainings(model_name, seed=SEED):
    num_epochs = 2
    print(f"\n{'='*50}")
    print(f"Scenario 4.3: Few-Shot Learning for {model_name}")
    print(f"{'='*50}")

    # --- 1. Train on 5% Data ---
    print("\n--- Training on 5% Data ---")
    model_5 = initialize_model(model_name)
    model_5, opt_5 = setup_linear_probe(model_5)
    train_loader_5 = get_few_shot_dataloaders(train_dataset, 0.05, seed=seed)
    _, history_5 = train_model(model_5, train_loader_5, val_loader, opt_5, num_epochs)

    # --- 2. Train on 20% Data ---
    print("\n--- Training on 20% Data ---")
    model_20 = initialize_model(model_name)
    model_20, opt_20 = setup_linear_probe(model_20)
    train_loader_20 = get_few_shot_dataloaders(train_dataset, 0.20, seed=seed)
    _, history_20 = train_model(model_20, train_loader_20, val_loader, opt_20, num_epochs)

    # --- 3. Train on 100% Data ---
    print("\n--- Training on 100% Data ---")
    model_100 = initialize_model(model_name)
    model_100, opt_100 = setup_linear_probe(model_100)
    train_loader_100 = get_few_shot_dataloaders(train_dataset, 1.00, seed=seed)
    _, history_100 = train_model(model_100, train_loader_100, val_loader, opt_100, num_epochs)

    # --- 4. Calculate Required Metrics ---
    # Extract final accuracies
    acc_val_5 = history_5['val_acc'][-1]
    acc_val_20 = history_20['val_acc'][-1]
    acc_val_100 = history_100['val_acc'][-1]

    acc_train_5 = history_5['train_acc'][-1]
    acc_train_20 = history_20['train_acc'][-1]
    acc_train_100 = history_100['train_acc'][-1]

    # Calculate Train-Val gaps
    gap_5 = acc_train_5 - acc_val_5
    gap_20 = acc_train_20 - acc_val_20
    gap_100 = acc_train_100 - acc_val_100

    # Calculate Relative Performance Drop
    rel_drop = (acc_val_100 - acc_val_5) / acc_val_100 if acc_val_100 > 0 else 0

    # --- 5. Print the Final Report ---
    print("\n" + "="*50)
    print(f"Few-Shot Analysis Results: {model_name}")
    print("="*50)

    print(f"1. Validation Accuracies:")
    print(f"   - 100% Data: {acc_val_100:.4f}")
    print(f"   - 20% Data:  {acc_val_20:.4f}")
    print(f"   - 5% Data:   {acc_val_5:.4f}")

    print(f"\n2. Training-Validation Gaps (Overfitting Assessment):")
    print(f"   - 100% Data Gap: {gap_100:.4f} (Train: {acc_train_100:.4f}, Val: {acc_val_100:.4f})")
    print(f"   - 20% Data Gap:  {gap_20:.4f} (Train: {acc_train_20:.4f}, Val: {acc_val_20:.4f})")
    print(f"   - 5% Data Gap:   {gap_5:.4f} (Train: {acc_train_5:.4f}, Val: {acc_val_5:.4f})")

    print(f"\n3. Relative Performance Drop:")
    print(f"   - Drop (100% vs 5%): {rel_drop:.4f} ({rel_drop*100:.2f}%)")
    print("="*50 + "\n")


## Experiment 4

In [82]:
def get_corrupted_val_loader(corruption_type, severity=None):
    base_transforms = [transforms.Resize((224, 224)), transforms.ToTensor()]

    if corruption_type == "gaussian":
        # Pixel-level Gaussian noise (sigma=0.05, 0.1, 0.2)
        base_transforms.append(v2.GaussianNoise(sigma=severity))
    elif corruption_type == "blur":
        base_transforms.append(transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5.)))
    elif corruption_type == "brightness":
        base_transforms.append(transforms.ColorJitter(brightness=severity))

    base_transforms.append(transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]))

    transform_corrupt = transforms.Compose(base_transforms)
    corrupted_dataset = datasets.ImageFolder(root=f"{DATA_DIR}/val", transform=transform_corrupt)
    val_dataset = Subset(corrupted_dataset, val_indices)
    return DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [83]:
def corruption_robustness_analysis(model_name, trained_model, clean_val_loader):
    print(f"\n{'='*60}")
    print(f"Scenario 4.4: Corruption Robustness Analysis for {model_name}")
    print(f"{'='*60}")

    # 1. Get baseline clean accuracy
    print("Evaluating baseline (clean) accuracy...")
    acc_clean = evaluate_model(trained_model, clean_val_loader, DEVICE)
    print(f"Baseline Clean Accuracy: {acc_clean:.4f}")

    # 2. Define the exact corruptions required by the assignment
    # The assignment asks for Gaussian (0.05, 0.1, 0.2), Motion blur, and Brightness shift
    corruptions_to_test = [
        ("Gaussian Noise (s=0.05)", "gaussian", 0.05),
        ("Gaussian Noise (s=0.1)",  "gaussian", 0.1),
        ("Gaussian Noise (s=0.2)",  "gaussian", 0.2),
        ("Motion Blur",             "blur",       None),
        ("Brightness Shift",        "brightness", 1.5) # Example severity for brightness
    ]

    print(f"\n{'-'*60}")
    print(f"{'Corruption Type':<25} | {'Acc':<6} | {'Error':<6} | {'Rel. Robustness':<15}")
    print(f"{'-'*60}")

    results = {}

    # 3. Evaluate each corruption
    for name, c_type, severity in corruptions_to_test:
        # Get the corrupted dataloader
        val_loader_corrupted = get_corrupted_val_loader(c_type, severity)

        # Pure evaluation, NO training
        acc_corrupted = evaluate_model(trained_model, val_loader_corrupted, DEVICE)

        # Calculate assignment metrics
        corruption_error = 1.0 - acc_corrupted
        relative_robustness = acc_corrupted / acc_clean if acc_clean > 0 else 0


        # Print table row
        print(f"{name:<25} | {acc_corrupted:.4f} | {corruption_error:.4f} | {relative_robustness:.4f}")

    print(f"{'-'*60}\n")

## Experiment 5

In [26]:
# Dictionary to store intermediate features
intermediate_features = {}

def get_features(name):
    def hook(model, input, output):
        # Flatten spatial dimensions (e.g., GAP) if it's a convolutional feature map
        if len(output.shape) == 4:
            output = torch.nn.functional.adaptive_avg_pool2d(output, (1, 1)).flatten(1)
        intermediate_features[name] = output.detach()
    return hook

def attach_hooks(model, model_name):

    if model_name == "resnet50":
        model.layer1.register_forward_hook(get_features('early'))
        model.layer3.register_forward_hook(get_features('middle'))
        model.layer4.register_forward_hook(get_features('final'))



## Confusion Matrix


In [27]:
def plot_confusion_matrix(model, val_loader, class_names, device):
    """
    Evaluates the model on the validation set and plots a confusion matrix.
    Since there are 30 classes, we use a larger figure size and disable
    individual cell annotations to keep it readable.
    """
    model.eval()
    all_preds = []
    all_labels = []

    # --- 1. Gather Predictions ---
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            # Move to CPU and convert to numpy for scikit-learn
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # --- 2. Compute Confusion Matrix ---
    cm = confusion_matrix(all_labels, all_preds)

    # --- 3. Plotting ---
    # A 15x12 figure is usually good for 30 classes
    plt.figure(figsize=(15, 12))

    # We set annot=False because numbers in a 30x30 grid get very cluttered.
    # If you want to see numbers, change to annot=True and add annot_kws={"size": 6}
    sns.heatmap(cm, annot=False, cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)

    plt.title('Confusion Matrix - Validation Set')
    plt.ylabel('True Class')
    plt.xlabel('Predicted Class')

    # Rotate x-axis labels so they don't overlap
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)

    plt.tight_layout()
    plt.show()
    plt.savefig('confusion_matrix.png', dpi=300)

## PCA



In [28]:
def plot_feature_embeddings(model, data_loader, class_names, device, method="pca", num_samples=1000):
    """
    Extracts features from the backbone and plots them using PCA or t-SNE.

    Args:
        method (str): "pca" or "tsne"
        num_samples (int): Max samples to plot (t-SNE gets very slow with too many points)
    """
    model.eval()
    features_list = []
    labels_list = []

    print(f"Extracting features for {method.upper()} visualization...")

    with torch.no_grad():
        for i, (inputs, labels) in enumerate(data_loader):
            inputs = inputs.to(device)

            # Extract features before the final classification head
            # timm's forward_features usually returns the unpooled feature map
            features = model.forward_features(inputs)

            # Global Average Pooling to flatten spatial dimensions if necessary
            if len(features.shape) == 4:
                features = torch.nn.functional.adaptive_avg_pool2d(features, (1, 1))
            features = features.flatten(1)

            features_list.append(features.cpu().numpy())
            labels_list.append(labels.cpu().numpy())

            # Stop early to avoid massive computation for t-SNE
            if (i + 1) * inputs.size(0) >= num_samples:
                break

    # Concatenate all batches
    X = np.concatenate(features_list, axis=0)[:num_samples]
    y = np.concatenate(labels_list, axis=0)[:num_samples]

    # Map label indices to actual string names for the legend
    y_names = [class_names[label] for label in y]

    print(f"Applying {method.upper()} dimensionality reduction on {X.shape[0]} samples...")

    # --- Dimensionality Reduction ---
    if method == "pca":
        reducer = PCA(n_components=2, random_state=42)
    elif method == "tsne":
        reducer = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
    else:
        raise ValueError("Method must be 'pca' or 'tsne'")

    X_reduced = reducer.fit_transform(X)

    # --- Plotting ---
    df = pd.DataFrame({
        'Component 1': X_reduced[:, 0],
        'Component 2': X_reduced[:, 1],
        'Class': y_names
    })

    plt.figure(figsize=(14, 10))
    sns.scatterplot(
        x='Component 1', y='Component 2',
        hue='Class',
        palette=sns.color_palette("husl", len(np.unique(y))),
        data=df,
        legend="full",
        alpha=0.7,
        s=60
    )

    plt.title(f'Feature Embeddings visualized with {method.upper()}')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., fontsize='small', ncol=2)
    plt.tight_layout()
    plt.show()
    plt.savefig(f'feature_embeddings_{method}.png', dpi=300, bbox_inches='tight')

## Training

In [84]:
def train_model(model, train_loader, val_loader, optimizer, num_epochs=2, experiment=2):
    """
    Standard training loop with automatic plotting of loss and accuracy curves.
    num_epochs=30 for full data, num_epochs=20 for few-shot scenarios.
    """
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()

    # Dictionaries to store metrics for plotting later
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }

    start_time = time.time()

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        epoch_grad_norms = []

        # --- Training Phase ---
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            loss.backward()
            # Record gradient norms just for the first batch of an epoch to keep overhead low
            if batch_idx == 0:
                epoch_grad_norms = track_gradient_norms(model)
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        epoch_train_loss = running_loss / total_train
        epoch_train_acc = correct_train / total_train


        epoch_val_loss, epoch_val_acc = evaluate_model(model, val_loader, DEVICE, criterion)

        # Save metrics
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)

        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.4f} | "
              f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}")

    elapsed_time = time.time() - start_time
    print(f"Training completed in {elapsed_time // 60:.0f}m {elapsed_time % 60:.0f}s")

    # --- Plotting Curves ---
    epochs = range(1, num_epochs + 1)
    plt.figure(figsize=(12, 5))

    # Plot 1: Loss vs. Epoch
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], label='Train Loss', marker='o')
    plt.plot(epochs, history['val_loss'], label='Val Loss', marker='o')
    plt.title('Loss vs. Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)

    # Plot 2: Accuracy vs. Epoch
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['train_acc'], label='Train Accuracy', marker='o')
    plt.plot(epochs, history['val_acc'], label='Val Accuracy', marker='o')
    plt.title('Accuracy vs. Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()
    # Optional: If you want to save it automatically for the report, you can add:
    # plt.savefig('training_curves.png', dpi=300)
    return model, history

In [85]:
 # --- Validation Phase ---
def evaluate_model(model, val_loader, DEVICE, criterion):
    model.eval()
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
    epoch_val_loss = running_val_loss / total_val
    epoch_val_acc = correct_val / total_val
    return epoch_val_loss, epoch_val_acc

## Main


In [ ]:
if __name__ == "__main__":
  model_name = "resnet50"
  # # 1. Linear Probe
  # trained_model, history = run_linear_probe(model_name)

  # class_names = base_val_dataset.classes
  # plot_confusion_matrix(trained_model, val_loader, class_names, DEVICE)
  # plot_feature_embeddings(trained_model, val_loader, class_names, DEVICE, method="pca", num_samples=1000)

  # # 2. Finetuning
  # run_finetuning_experiments(model_name)

  # 3. Few-shot strategies
  # run_few_shot_trainings(model_name)

  # 4. Corruption robustness
  model = initialize_model(model_name)
  model, optimizer = setup_linear_probe(model)
  trained_model, _ = train_model(model, train_loader, val_loader, optimizer, num_epochs=2)
  corruption_robustness_analysis(model_name, trained_model, val_loader)

Epoch [1/2] Train Loss: 2.9778, Train Acc: 0.3545 | Val Loss: 2.5923, Val Acc: 0.5818


In [ ]:
# plot_confusion_matrix(trained_model, val_loader, class_names, DEVICE)

In [ ]:
# plot_feature_embeddings(trained_model, val_loader, class_names, DEVICE, method="pca", num_samples=1000)